# The Standard Library

[ACME lab](https://labs.acme.byu.edu/PythonEssentials/StandardLibrary/StandardLibrary.html) · Volume 2 · Vikunja task #66

This notebook covers all five problems. The solutions are in `standard_library.py` and `calculator.py`; `box.py` is the unmodified helper supplied with the lab. Every code cell runs and saves visible output.

## Built-ins and namespaces

Functions such as `min`, `max`, `len`, `sum`, `round`, and `abs` are available without importing a module. A Python namespace maps names to objects; assigning a second name to a mutable object makes both names point to the same object.

**Problem 1.** Return minimum, maximum, and arithmetic mean from a nonempty list using one return statement and built-in functions.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys
sys.path.insert(0, str(Path.cwd()))

spec = importlib.util.spec_from_file_location("standard_library", Path("standard_library.py"))
solutions = importlib.util.module_from_spec(spec)
spec.loader.exec_module(solutions)
for values in ([1, 2, 3, 4], [-5, 10, 7], [42]):
    result = solutions.min_max_avg(values)
    assert result == (min(values), max(values), sum(values) / len(values))
    print(f"{values} -> min={result[0]}, max={result[1]}, average={result[2]}")

[1, 2, 3, 4] -> min=1, max=4, average=2.5
[-5, 10, 7] -> min=-5, max=10, average=4.0
[42] -> min=42, max=42, average=42.0


## Mutability

A list, set, or dictionary can change in place; an integer, string, or tuple cannot. Assigning a new value to one name for an immutable object leaves the other name pointing to the old value.

**Problem 2.** Alias each requested type, alter one name, and compare the two results.

In [2]:
a = 1; b = a; a += 1
print("int:", a, b, "immutable"); assert a != b
a = "cat"; b = a; a += "s"
print("str:", a, b, "immutable"); assert a != b
a = [1]; b = a; a.append(2)
print("list:", a, b, "mutable"); assert a == b
a = (1,); b = a; a += (2,)
print("tuple:", a, b, "immutable"); assert a != b
a = {1}; b = a; a.add(2)
print("set:", a, b, "mutable"); assert a == b
print("Conclusion: int, str, tuple are immutable; list, set are mutable.")

int: 2 1 immutable
str: cats cat immutable
list: [1, 2] [1, 2] mutable
tuple: (1, 2) (1,) immutable
set: {1, 2} {1, 2} mutable
Conclusion: int, str, tuple are immutable; list, set are mutable.


## Modules and imports

A module is a Python file that can be imported. Code in an `if __name__ == "__main__":` block runs only when the module is executed as a script. Standard library modules include `math`, `itertools`, `random`, `sys`, and `time`.

**Problem 3.** The separate `calculator.py` exposes `sum`, `product`, and imported `sqrt`. The hypotenuse function uses only those names.

In [3]:
import calculator
assert calculator.sum(2, 3) == 5
assert calculator.product(2, 3) == 6
assert solutions.hypotenuse(3, 4) == 5
silent = subprocess.run([sys.executable, "calculator.py"], capture_output=True, text=True, check=True)
assert silent.stdout == "" and silent.stderr == ""
print("calculator.sum(2, 3) =", calculator.sum(2, 3))
print("calculator.product(2, 3) =", calculator.product(2, 3))
print("hypotenuse(3, 4) =", solutions.hypotenuse(3, 4))
print("calculator.py runs without side effects:", silent.stdout == "")

calculator.sum(2, 3) = 5
calculator.product(2, 3) = 6
hypotenuse(3, 4) = 5.0
calculator.py runs without side effects: True


## Combinations and the power set

`itertools.combinations(A, k)` yields every unordered choice of `k` elements. Combining the results for every `k` from zero to the input size gives the power set. A set of sets is impossible because sets are mutable and therefore unhashable.

**Problem 4.** Return the power set as a list of sets, including `set()` and the whole input.

In [4]:
subsets = solutions.power_set(["a", "b", "c"])
assert len(subsets) == 8
assert set() in subsets and {"a", "b", "c"} in subsets
assert {"a", "c"} in subsets
print("Power set of {a, b, c}:", subsets)
print("Number of subsets:", len(subsets))

Power set of {a, b, c}: [set(), {'a'}, {'b'}, {'c'}, {'b', 'a'}, {'c', 'a'}, {'c', 'b'}, {'c', 'b', 'a'}]
Number of subsets: 8


## Randomness, command line arguments, and time

The `random` module simulates dice; `sys.argv` holds command line arguments; `time.monotonic()` measures elapsed time reliably. The supplied `box.py` validates whether a roll has a matching combination and parses the player's choices.

**Problem 5.** `standard_library.py NAME SECONDS` starts Shut the Box only with exactly two user arguments. The game rolls two dice while the remaining numbers total more than six, then one die. It repeats an invalid choice without rolling again, ends on a win, impossible roll, or timeout, and reports score and elapsed time.

In [5]:
import contextlib
import io

rolls = iter([6, 6, 3, 6, 1, 6, 1, 1, 5, 6, 4])
answers = iter(["3 3", "3 9", "8 1", "7", "2", "5 6", "4"])
roll_count = [0]
def fixed_die(low, high):
    roll_count[0] += 1
    return next(rolls)
def fixed_input(prompt):
    answer = next(answers)
    print(prompt + answer)
    return answer
output = io.StringIO()
with contextlib.redirect_stdout(output):
    won, score = solutions.play_game("LuckyDuke", 60, input_fn=fixed_input, randint=fixed_die, clock=lambda: 0.0)
assert won and score == 0 and roll_count[0] == 11
assert "Invalid choice; try again." in output.getvalue()
assert "Congratulations" in output.getvalue()
print(output.getvalue())
print("Verified six turns, invalid-input retry, one-die final turn, and a win.")

Numbers left: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Roll: 12
Seconds left: 60.0
Numbers to eliminate: 3 3
Invalid choice; try again.
Numbers to eliminate: 3 9
Numbers left: [1, 2, 4, 5, 6, 7, 8]
Roll: 9
Seconds left: 60.0
Numbers to eliminate: 8 1
Numbers left: [2, 4, 5, 6, 7]
Roll: 7
Seconds left: 60.0
Numbers to eliminate: 7
Numbers left: [2, 4, 5, 6]
Roll: 2
Seconds left: 60.0
Numbers to eliminate: 2
Numbers left: [4, 5, 6]
Roll: 11
Seconds left: 60.0
Numbers to eliminate: 5 6
Numbers left: [4]
Roll: 4
Seconds left: 60.0
Numbers to eliminate: 4
Score for player LuckyDuke: 0 points
Time played: 0.00 seconds
Congratulations! You shut the box!

Verified six turns, invalid-input retry, one-die final turn, and a win.


In [6]:
ticks = iter([0.0, 0.0, 0.0, 3.0, 3.0, 3.0])
timeout_output = io.StringIO()
with contextlib.redirect_stdout(timeout_output):
    won, score = solutions.play_game("TimedOut", 2, input_fn=lambda _: "3 9", randint=lambda *_: 6, clock=lambda: next(ticks))
assert not won and score == 45
assert "Time is up!" in timeout_output.getvalue()
missing_args = subprocess.run([sys.executable, "standard_library.py"], capture_output=True, text=True, check=True)
assert missing_args.stdout == ""
print(timeout_output.getvalue())
print("Missing command-line arguments start no game:", missing_args.stdout == "")

Numbers left: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Roll: 12
Seconds left: 2.0
Time is up!
Score for player TimedOut: 45 points
Time played: 3.00 seconds
Better luck next time!

Missing command-line arguments start no game: True
